In [2]:
import torch
if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

tensor([1.], device='mps:0')


In [3]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import cv2
from PIL import Image
import pandas as pd
import numpy as np

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset

%load_ext autoreload
%autoreload 2

ModuleNotFoundError: No module named 'pytorch_lightning'

In [ ]:
path_to_dataset = Path("../../data/01_raw/sneakers-dataset")

In [ ]:
# Изображения хранятся в директории так, что каждой модели кроссовок соответствует
# своя папка. Чтобы можно было рассчитывать статистики, мы собираем в датафрейм относительный
# путь до каждого из изображений, и рассчитываем агрегаты на основе этого датафрейма
from src.data.utils.eda_utils import directory_to_dataframe

path_to_dataset = Path("../../data/01_raw/sneakers-dataset")
df = directory_to_dataframe(path_to_dataset)
df.head()

,path,sneaker_class
0,nike_air_force_1_high/0036.jpg,nike_air_force_1_high
1,nike_air_jordan_1_high/0026.jpg,nike_air_jordan_1_high
2,converse_one_star/0025.jpg,converse_one_star
3,reebok_classic_leather/0004.jpg,reebok_classic_leather
4,nike_air_jordan_11/0014.jpg,nike_air_jordan_11


In [ ]:
train_df_pre = pd.read_csv('../../data/01_raw/train_images.csv')

display(train_df_pre.head(), train_df_pre.shape)

Bad images len: 12
Dataframe size: 5796


(4636, 2)

In [ ]:
test_df = pd.read_csv('../../data/01_raw/test_images.csv')

display(test_df.head(), test_df.shape)

,path,sneaker_class
0,adidas_forum_low/0026.jpg,adidas_forum_low
1,nike_air_jordan_4/0078.jpg,nike_air_jordan_4
2,yeezy_boost_350_v2/0018.jpg,yeezy_boost_350_v2
3,adidas_superstar/0028.jpg,adidas_superstar
4,adidas_forum_high/0091.jpg,adidas_forum_high


(1160, 2)

In [ ]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.25,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,path,sneaker_class
4439,converse_chuck_70_low/0086.jpg,converse_chuck_70_low
2029,nike_dunk_low/0038.jpg,nike_dunk_low
3641,vans_old_skool/0059.jpg,vans_old_skool
3041,adidas_stan_smith/0001.jpg,adidas_stan_smith
142,nike_air_jordan_1_high/0009.jpg,nike_air_jordan_1_high


(3477, 2)

,path,sneaker_class
455,new_balance_327/0028.jpg,new_balance_327
705,reebok_club_c_85/0070.jpg,reebok_club_c_85
3532,asics_gel-lyte_iii/0017.jpg,asics_gel-lyte_iii
4587,nike_dunk_high/0050.jpg,nike_dunk_high
1185,reebok_classic_leather/0019.jpg,reebok_classic_leather


(1159, 2)

,path,sneaker_class
0,adidas_forum_low/0026.jpg,adidas_forum_low
1,nike_air_jordan_4/0078.jpg,nike_air_jordan_4
2,yeezy_boost_350_v2/0018.jpg,yeezy_boost_350_v2
3,adidas_superstar/0028.jpg,adidas_superstar
4,adidas_forum_high/0091.jpg,adidas_forum_high


(1160, 2)

In [4]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

NameError: name 'train_df' is not defined

In [5]:
train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

NameError: name 'A' is not defined

In [6]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

NameError: name 'ImageDataset' is not defined

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=8,
    pin_memory=False,
    persistent_workers=True,
)

NameError: name 'DataLoader' is not defined

# Baseline CNN

In [19]:
# model = LitCNN(num_classes=df["sneaker_class"].nunique())
resnet_model = LitResNet18(num_classes=df["sneaker_class"].nunique(),
                           lr=0.001)

trainer = pl.Trainer(
    max_epochs=20,
    accelerator="auto",
    devices=1,
    logger=pl.loggers.TensorBoardLogger("lightning_logs", name="cnn_baseline"),
    log_every_n_steps=1
)

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores


In [12]:
import warnings 
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

resnet_trainer.fit(resnet_model, train_loader, val_loader)

┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 11.2 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 69                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pyt
ree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and 
treespec.is_leaf()` instead.

`Trainer.fit` stopped: `max_epochs=10` reached.


In [13]:
resnet_pred_batches = resnet_trainer.predict(resnet_model, dataloaders=[test_loader])

/Users/a.r.makarenko/Documents/hse/sneakers-hse/.venv/lib/python3.13/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

In [14]:
y_pred = torch.cat(pred_batches).cpu().numpy()

In [15]:
y_true = [x[1].numpy().item() for x in test_dataset]

In [16]:
y_pred_resnet = torch.cat(resnet_pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred_resnet))

              precision    recall  f1-score   support

           0       1.00      0.37      0.54        30
           1       0.85      0.61      0.71        18
           2       0.54      0.73      0.62        30
           3       0.85      0.58      0.69        19
           4       0.54      0.50      0.52        14
           5       0.88      0.52      0.65        29
           6       1.00      0.26      0.42        19
           7       0.60      0.87      0.71        30
           8       0.62      0.72      0.67        18
           9       0.70      0.47      0.56        15
          10       0.42      0.67      0.51        30
          11       0.64      0.47      0.54        15
          12       0.67      0.11      0.19        18
          13       1.00      0.67      0.80        30
          14       0.31      0.91      0.47        22
          15       0.86      0.63      0.73        30
          16       0.84      0.53      0.65        30
          17       0.23    

In [17]:
trainer.callback_metrics

{}